## Import de la base de données contenant les IDMuseofiles

Nous souhaitons importer cette première base de données pour en extraire les "identifiants muséofiles" qui sont absent dans les bases de données de fréquentation.

In [1]:
import pandas as pd

from fonctions import (
    ajouter_colonne_statut_precis, 
    nettoyer_frequentation
    )


# Import des données individuelles
url_individu = "https://object.data.gouv.fr/ministere-culture/FREQ_MUSEES/ENTREES_ET_CATEGORIES_DE_PUBLIC.csv"


df_individu = pd.read_csv(url_individu, sep=';')
df_individu = (
    df_individu
    .loc[:,['IDPatrimostat', 'IDMuseofile']]
    )
df_individu = (
    df_individu
    .rename(columns={'IDPatrimostat' : 'REF DU MUSEE'})
    )


# On supprime les lignes dupliquées
df_individu = (
    df_individu
    .drop_duplicates(subset=["REF DU MUSEE"])
    )

## Import et nettoyage de la base de données principale

Import des bases de données suivantes qui vont nous servir de base d'analyse. Elles contiennent toutes les fréquentations.

In [2]:
# Import des données totales
url = "https://static.data.gouv.fr/resources/frequentation-des-musees-de-france-1/20250827-121955/frequentation-totale-mdf-2001-a-2016-data-def9.xlsx"
df_totale = pd.read_excel(url, sheet_name=None)


# Pour voir les noms des feuilles disponibles
print(df_totale.keys())


# Pour accéder à une table précise
df_freq_totale = df_totale['FREQUENTATION TOTALE']


# On veut ajouter les idmuseofiles pour la suite de l'étude
df_freq_totale = (
    df_freq_totale
    .merge(df_individu, on="REF DU MUSEE", how="left")
    )  
#print(df_freq_totale.head())


df_freq_gratuite = df_totale['FREQ GRATUITE']
#print(df_freq_gratuite.head())


df_freq_payante = df_totale['FREQ PAYANTE']
#print(df_freq_payante.head())

dict_keys(['FREQUENTATION TOTALE', 'FREQ GRATUITE', 'FREQ PAYANTE'])


Les bases de données de fréquentation payante et gratuite contiennent les même musées. En fait, les musées proposent divers types de visites : payantes et gratuites.
Au total, il y a 1241 musées.

In [3]:
nb_musees = len(df_freq_totale)
nb_musees

1241

Dans la deuxième base, le tableau "payant" n'a pas les mêmes noms de variables que les deux autres.

In [4]:
df_freq_payante = (
    df_freq_payante
    .rename(columns={"REF MUSEE": "REF DU MUSEE",})
    )
df_freq_totale = (
    df_freq_totale
    .rename(columns={"NEW REGIONS": "NOMREG",})
    )
df_freq_gratuite = (
    df_freq_gratuite
    .rename(columns={"NEW REGIONS": "NOMREG",})
    )

Nous souhaitons transformer la deuxième base de données pour avoir une seule variable année.

In [5]:

# Colonnes identifiantes à conserver
id_vars = (
    ["REF DU MUSEE", 
    "NOMREG", 
    "NOM DU MUSEE", 
    "VILLE", 
    "Fréquentation"]
    )
id_vars_1 = (
    ["REF DU MUSEE", 
    "NOMREG", 
    "NOM DU MUSEE", 
    "VILLE", 
    "Fréquentation", 
    "IDMuseofile"]
    )


# Colonnes années
annee_vars = [str(y) for y in range(2001, 2017)]


# Passage en format long
df_freq_totale = df_freq_totale.melt(
    id_vars = id_vars_1,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)


df_freq_gratuite = df_freq_gratuite.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)


df_freq_payante = df_freq_payante.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

Nous souhaitons connaître le nombre de musée n'ayant jamais communiqué leur fréquentation entre 2001 et 2016.

In [6]:
# Quelles valeurs, autres que numériques, peut prendre la fréquentation?
valeurs_non_numeriques = (
    df_freq_totale.loc[
        pd.to_numeric(df_freq_totale["frequentation"], errors="coerce")
        .isna(),
        "frequentation"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)


print(sorted(valeurs_non_numeriques))

['F', 'NC', "Retrait d'appelaltion", "Retrait d'appellation", 'SO', 'Transfert à Marseille - MUCEM', 'Transfert à Nice']


Combien de musées présentent ces modalités au moins une fois ?

In [7]:
freq = df_freq_totale["frequentation"]


# On garde les lignes où la valeur est NA ou n'est pas un nombre
ligne = freq.isna() | pd.to_numeric(freq, errors="coerce").isna()


# On crée une colonne avec les modalités
df_temp = (
    df_freq_totale
    .loc[ligne, ["REF DU MUSEE", "frequentation"]]
    .copy()
    )


# On remplace les NA par le texte "NA"
df_temp["frequentation"] = (
    df_temp["frequentation"]
    .fillna("NA")
    )


# On compte le nombre de musées différents pour chaque modalité
resultat = (
    df_temp
    .groupby("frequentation")["REF DU MUSEE"]
    .nunique()
    )

print(resultat)

frequentation
F                                340
NA                               230
NC                               224
Retrait d'appelaltion              2
Retrait d'appellation              2
SO                                38
Transfert à Marseille - MUCEM      1
Transfert à Nice                   1
Name: REF DU MUSEE, dtype: int64


Nous souhaitons maintenant connaître le nombre de musées présentant, toutes les années, une de ces modalités non numériques.

In [8]:
# On marque les lignes où la fréquentation est NA ou non numérique
df_freq_totale["speciale"] = freq.isna() | pd.to_numeric(freq, errors="coerce").isna()


# Pour chaque musée, on regarde si toutes les lignes sont spéciales
resultat_toutes_annees = (
    df_freq_totale
    .groupby("REF DU MUSEE")["speciale"]
    .all()
    )


# Nombre de musées concernés
nb_musees_toutes_annees = resultat_toutes_annees.sum()
pourcentage = (nb_musees_toutes_annees / nb_musees) * 100


print(
    f"{nb_musees_toutes_annees} musées n'ont aucune fréquentation "
    f"renseignée entre 2001 et 2016, soit {pourcentage:.2f}%."
)

79 musées n'ont aucune fréquentation renseignée entre 2001 et 2016, soit 6.37%.


On peut supprimer les lignes vides (non réponse totale)

In [9]:
# Suppression des lignes où toutes les colonnes sont vides
df_freq_totale = df_freq_totale.dropna(how='all')
df_freq_gratuite = df_freq_gratuite.dropna(how='all')
df_freq_payante = df_freq_payante.dropna(how='all')

On souhaite, pour la suite des analyses, créer deux colonnes : l'une avec les statuts du musée et l'autre avec seulement des variables numériques (on remplace par NaN les valeurs non numériques)

In [10]:
# Application de la fonction à tes 3 DataFrames
df_freq_totale = nettoyer_frequentation(df_freq_totale)
df_freq_payante = nettoyer_frequentation(df_freq_payante)
df_freq_gratuite = nettoyer_frequentation(df_freq_gratuite)


# Ajout de la colonne statut
df_freq_totale = ajouter_colonne_statut_precis(df_freq_totale)
df_freq_payante = ajouter_colonne_statut_precis(df_freq_payante)
df_freq_gratuite = ajouter_colonne_statut_precis(df_freq_gratuite)


# Distribution des statuts
print(
    df_freq_totale["Statut"]
    .value_counts(dropna=False)
    )

Statut
Ouvert                   15779
Fermé                     2249
NA                        1757
Sans Objet                  65
Retrait d'appellation        4
Transfert                    2
Name: count, dtype: int64


## Import de la base de données répertoriant les types de musées

On souhaite maintenant connaître la thématique des musées. Pour ce faire, nous importons une nouvelle base de données.

In [11]:
url_types_musees = "https://object.data.gouv.fr/ministere-culture/POP/museofile.csv"


df_types_musees = pd.read_csv(url_types_musees, sep='|')
df_types_musees = df_types_musees[['Identifiant','Domaine_thematique']]
df_types_musees = (
    df_types_musees
    .rename(columns={'Identifiant' : 'IDMuseofile'})
    )

Nous ajoutons les informations sur les thématiques du musées à la bases de données sur les fréquentations

In [12]:
df_freq_totale = (
    df_freq_totale
    .merge(df_types_musees, on="IDMuseofile", how="left")
    )

La colonne Domaine_thematique peut contenir plusieurs thématiques : "Archéologie;Arts décoratifs;Art moderne" par exemple. Nous aimerions créer des colonnes binaire pour chaque thématique (1 si le musées aborde cette thématique 0 sinon).
Il s'agit aussi de corriger les différentes coquilles dans l'écriture des thématiques.

In [13]:
# On sépare les thématiques et on crée les colonnes binaires
colonnes_binaires = (
    df_freq_totale['Domaine_thematique']
    .str
    .get_dummies(sep=';')
    )
print(colonnes_binaires.columns) 
# On remarque que des thématiques sont présentes deux fois car mal écrites


# Nettoyage de colonnes binaires
colonnes_binaires.columns = (
    colonnes_binaires.columns
    .str
    .strip()
    )
# enlever les espaces au début/à la fin des noms de colonnes


# dictionnaire de fusion des doublons / variantes
fusions = {
    'Amerique': ['Amérique'],
    'Afrique': ['Afrique'],
    'Archeologie': ['Archéologie', 'archéologie'],
    'Beaux_arts': ['Beaux-Arts', 'Beaux-arts', 'beaux-arts'],
    'Arts_decoratifs': ['Arts décoratifs', 'arts décoratifs'],
    'Ethnologie': ['Ethnologie', 'Ethnonolgie', 'ethnologie'],
    'Histoire': ['Histoire', 'histoire'],
    'Litterature': ['Littérature', 'littérature'],
    'Photographie': ['Photographie', 'photographie'],
    'Oceanie': ['OCéanie', 'Océanie'],
    'Sciences_techniques': ['Sciences et techniques', 'Scieinces et techniques'],
}


# fusion des colonnes binaires proches en prenant le max ligne par ligne
for nouvelle_col, anciennes_cols in fusions.items():
    cols_presentes = [c for c in anciennes_cols if c in colonnes_binaires.columns]
    if cols_presentes:
        colonnes_binaires[nouvelle_col] = colonnes_binaires[cols_presentes].max(axis=1)
        colonnes_binaires.drop(columns=cols_presentes, inplace=True)


# Rattacher à df_freq_totale
df_freq_totale = (
    pd
    .concat([df_freq_totale, colonnes_binaires], axis=1)
    )

Index([' Amérique', 'Afrique', 'Amérique', 'Archéologie',
       'Art moderne et contemporain', 'Arts de l'Islam', 'Arts décoratifs',
       'Asie', 'Beaux-Arts', 'Beaux-arts', 'Couturier', 'Design', 'Egyptien',
       'Esclavage, société de plantation, histoire de La Réunion, colonisation, industrie sucrière',
       'Ethnologie', 'Ethnonolgie', 'Gallo-romain', 'Grec', 'Histoire',
       'Imprimé', 'Inde', 'Jeux', 'Littérature', 'Militaria', 'Mode',
       'Mode et textile', 'Musique', 'Musique - chant - danse',
       'Mémoire de l'esclavage', 'Numismatique', 'OCéanie', 'Océanie',
       'Peinture', 'Photographie', 'Protohistoire', 'Scieinces et techniques',
       'Sciences', 'Sciences de la nature', 'Sciences et techniques',
       'Sciences fondamentales', 'Sciences naturelles',
       'Technique et industrie', 'Techniques', 'Textile', 'archéologie',
       'archéologie du bâti', 'arts décoratifs', 'beaux-arts', 'ethnologie',
       'histoire', 'littérature', 'musée de société', '